In [49]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [50]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

In [51]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [52]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [53]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [54]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [55]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [56]:
nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

# Fetch candle data for each instrument and each timeframe
nifty_2m = fetch_train_candle_data(10, nf_index_symbol, 2)
nifty_5m = fetch_train_candle_data(10, nf_index_symbol, 5)
nifty_15m = fetch_train_candle_data(10, nf_index_symbol, 15)

bnf_2m = fetch_train_candle_data(10, bnf_index_symbol, 2)
bnf_5m = fetch_train_candle_data(10, bnf_index_symbol, 5)
bnf_15m = fetch_train_candle_data(10, bnf_index_symbol, 15)

# Clean and remove duplicate datetimes (if any)
nifty_2m = nifty_2m.drop_duplicates(subset='datetime', keep='first')
nifty_5m = nifty_5m.drop_duplicates(subset='datetime', keep='first')
nifty_15m = nifty_15m.drop_duplicates(subset='datetime', keep='first')

bnf_2m = bnf_2m.drop_duplicates(subset='datetime', keep='first')
bnf_5m = bnf_5m.drop_duplicates(subset='datetime', keep='first')
bnf_15m = bnf_15m.drop_duplicates(subset='datetime', keep='first')

In [57]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        """
        self.df = df.copy()

        # We no longer need an htf_map since higher timeframe features are removed
        # self.htf_map = { ... }  # Removed

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates/missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute a few standard technical indicators on the base timeframe:
        RSI, MACD, Bollinger Bands, ATR, etc.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR (for baseline volatility measure)
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_candlestick_patterns(self):
        """
        Detect a few common candlestick patterns.
        We store them as numeric columns (0 = not present, 1 = bullish pattern, -1 = bearish pattern, etc.)
        so that the pipeline remains consistent with numeric-only scaling.
        """
        self.df.sort_index(inplace=True)

        # Example: Doji detection (very simplistic)
        # We check if the open and close are nearly the same
        body_size = (self.df['close'] - self.df['open']).abs()
        candle_range = self.df['high'] - self.df['low']
        self.df['candle_doji'] = np.where(
            (body_size <= 0.1 * candle_range), 1.0, 0.0
        )

        # Example: Engulfing pattern (bullish and bearish)
        # - Bullish Engulfing: today's open < yesterday's close, today's close > yesterday's open
        # - Bearish Engulfing: today's open > yesterday's close, today's close < yesterday's open
        # We shift the open/close by 1 day to compare
        self.df['open_shift1'] = self.df['open'].shift(1)
        self.df['close_shift1'] = self.df['close'].shift(1)

        cond_bull_engulf = (self.df['open'] < self.df['close_shift1']) & (self.df['close'] > self.df['open_shift1'])
        cond_bear_engulf = (self.df['open'] > self.df['close_shift1']) & (self.df['close'] < self.df['open_shift1'])

        self.df['candle_engulf'] = np.where(cond_bull_engulf, 1.0,
                                    np.where(cond_bear_engulf, -1.0, 0.0))

        # Drop helper columns to keep dataset clean
        self.df.drop(['open_shift1', 'close_shift1'], axis=1, inplace=True)

        return self

    def add_swing_points(self, left_bars=3, right_bars=3):
        """
        Detect local swing highs and lows.
        left_bars/right_bars define how many bars on each side must be lower/higher (for highs/lows).
        """
        self.df.sort_index(inplace=True)

        # For each row, check if 'high' is greater than the preceding and following N bars
        rolling_high = self.df['high'].rolling(left_bars+1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars+1, center=False).min()

        # We'll do a simplistic approach: shift rolling computations accordingly
        # A more robust approach might be to iterate row by row, but we'll keep it vectorized for brevity.

        # Swing High
        # Compare current high to future bars as well, so we do a forward rolling
        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars+1, center=False).max()
        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        # Swing Low
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars+1, center=False).min()
        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        """
        Add Donchian channels, breakout flags, and a simple measure of range expansion.
        """
        self.df.sort_index(inplace=True)

        # Donchian Channels
        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Breakout flags
        #  1.0 if close > donchian_high, -1.0 if close < donchian_low, else 0.0
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion/compression example
        # Rolling std of (high - low)
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        """
        Add additional momentum-based indicators like Stochastics, ADX, CCI, etc.
        (We already have RSI, MACD in add_base_timeframe_features, but you can unify them here if you prefer.)
        """
        self.df.sort_index(inplace=True)

        # Stochastic
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX (Average Directional Movement Index)
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI (Commodity Channel Index)
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        """
        Add additional volatility-based features, e.g. historical volatility, Z-score of returns, etc.
        """
        self.df.sort_index(inplace=True)

        # Historical volatility (using close-to-close log returns)
        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)  # Annualized approximation

        # Z-Score of Price Changes over 'window'
        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        """
        Add volume-based features if 'volume' is present.
        """
        if 'volume' not in self.df.columns:
            return self  # No volume data, skip

        self.df.sort_index(inplace=True)

        # On-Balance Volume
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spike detection:
        # Compare current volume to rolling average
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP over the day (if you have daily or session logic, you'd groupby date)
        # Here, for a simpler approach, we do a rolling 1-day approximation:
        # This might not be perfectly accurate if you have continuous 24h data, but it’s a start.
        # Usually you’d reset VWAP each day or each session.
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        # Drop helper columns
        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        """
        Classify each bar as 'trending' or 'ranging' (0 or 1) based on ADX, or volatility-based thresholds, etc.
        Keep it numeric for scaling (e.g., 0.0 or 1.0).
        """
        self.df.sort_index(inplace=True)

        # If ADX above 25 => trending, else ranging
        # (25 is just a typical reference threshold; pick whichever fits your strategy)
        if 'adx' not in self.df.columns:
            # If ADX wasn't yet computed, do it quickly
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        # Alternatively, you could have multiple columns for different regime classifications
        return self

    def add_time_features(self):
        """
        Extract cyclical or numeric time features (hour of day, day of week).
        We ensure they are numeric (float).
        """
        self.df.sort_index(inplace=True)

        # Because 'datetime' is now our index, we can access it via self.df.index
        # Create columns for hour, day of week
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        # Cyclical encoding for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        # Cyclical encoding for day_of_week
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        # You can drop raw hour/day_of_week if you prefer only cyclical features
        # self.df.drop(['hour', 'day_of_week'], axis=1, inplace=True)

        return self

    def add_adaptive_targets_and_stops(self):
        """
        Instead of fixed (2 * ATR) / (1 * ATR), adapt based on current volatility or regime.
        For demonstration, we’ll say:
            - If regime is trending, target = 3 * ATR, stop = 1.5 * ATR
            - If regime is ranging, target = 1.5 * ATR, stop = 1 * ATR
        This is just an example logic.
        """
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # If we haven't computed 'regime_trend', do so
        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        # Example adaptation
        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def run_pipeline(self):
        """
        Run all steps except higher timeframe computations (now removed).
        You can rearrange the order as desired.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .add_candlestick_patterns()
             .add_swing_points()
             .add_range_breakout_features()
             .add_momentum_features()
             .add_volatility_features()
             .add_volume_features()            # only applies if 'volume' exists
             .add_regime_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
        )
        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        We'll also ensure that all columns except 'Signal' are float with 2 decimal places.
        """
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        # Ensure 'Signal' is int; everything else float with 2 decimals
        if 'Signal' in self.df.columns:
            sig_col = self.df['Signal'].astype(int)
            self.df.drop(columns=['Signal'], inplace=True)
        else:
            sig_col = None

        # Convert all remaining columns to float
        for col in self.df.columns:
            self.df[col] = self.df[col].astype(float)

        # Round to 2 decimals
        self.df = self.df.round(2)

        # Reinsert Signal column if it exists
        if sig_col is not None:
            self.df['Signal'] = sig_col

        return self.df

# Nifty
nf_train_pipeline_2m = FullFeaturePipeline(nifty_2m)
nf_train_pipeline_2m.run_pipeline()
df_nifty_2m = nf_train_pipeline_2m.get_processed_df()

nf_train_pipeline_5m = FullFeaturePipeline(nifty_5m)
nf_train_pipeline_5m.run_pipeline()
df_nifty_5m = nf_train_pipeline_5m.get_processed_df()

nf_train_pipeline_15m = FullFeaturePipeline(nifty_15m)
nf_train_pipeline_15m.run_pipeline()
df_nifty_15m = nf_train_pipeline_15m.get_processed_df()

# Bank Nifty
bnf_train_pipeline_2m = FullFeaturePipeline(bnf_2m)
bnf_train_pipeline_2m.run_pipeline()
df_bnf_2m = bnf_train_pipeline_2m.get_processed_df()

bnf_train_pipeline_5m = FullFeaturePipeline(bnf_5m)
bnf_train_pipeline_5m.run_pipeline()
df_bnf_5m = bnf_train_pipeline_5m.get_processed_df()

bnf_train_pipeline_15m = FullFeaturePipeline(bnf_15m)
bnf_train_pipeline_15m.run_pipeline()
df_bnf_15m = bnf_train_pipeline_15m.get_processed_df()

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Version 1

In [58]:

# %%
# ===========================================
# 1) NECESSARY IMPORTS AND DEPENDENCIES
# ===========================================
import gym
import numpy as np
import pandas as pd
import random

# %%
# ======================================================================================
# 2) SINGLEINSTRUMENTTRADINGENV WITH TARGET/STOPLOSS + INSTRUMENT-SPECIFIC QUANTITY LOGIC
# ======================================================================================
class SingleInstrumentTradingEnv(gym.Env):
    """
    An environment that:
      1) Chooses an instrument.
      2) Chooses a timeframe for that instrument.
      3) Loads the corresponding DataFrame (which can have a unique length).
      4) Trains or tests a trading strategy with capital-based logic, slippage, etc.

    Enhancements here:
      - "Target" and "StopLoss" logic:
          * If basic PnL >= 0 => realized = +Target * quantity
          * Else => realized = -StopLoss * quantity
      - instrument-specific quantities (e.g. NIFTY=75, BANKNIFTY=30) stored in config["quantity_dict"]
      - capital gains/losses multiplied by the instrument's quantity

    Supports:
      - Different lengths of data per timeframe
      - Brokerage and slippage
      - Sharpe-based reward component
      - Flexible reward transformations (raw, pct, log, clip)
      - Observations can be the entire DataFrame row as float32
    """

    def __init__(self, config):
        super(SingleInstrumentTradingEnv, self).__init__()

        # ----------------
        # 1) CONFIG / SETUP
        # ----------------
        self.config = config

        # data_dict => { instrument_name: { timeframe: DataFrame, ... }, ... }
        self.data_dict = self.config.get("data_dict")
        if not self.data_dict or len(self.data_dict) == 0:
            raise ValueError("config['data_dict'] must have at least one instrument entry.")

        # Brokerage cost dict => { instrument: cost }
        self.brokerage_dict = self.config.get("brokerage_dict", {})
        for inst in self.data_dict.keys():
            if inst not in self.brokerage_dict:
                raise ValueError(f"brokerage_dict missing entry for '{inst}'.")

        # Instrument/timeframe selection approach
        self.instrument_selection = self.config.get("instrument_selection", "random")
        self.timeframe_selection = self.config.get("timeframe_selection", "random")

        # Environment parameters
        self.initial_capital_multiplier = self.config.get("initial_capital_multiplier", 3.0)
        self.flip_position = self.config.get("flip_position", True)
        self.track_unrealized_in_reward = self.config.get("track_unrealized_in_reward", True)
        self.unrealized_reward_weight = self.config.get("unrealized_reward_weight", 0.3)
        self.reward_mode = self.config.get("reward_mode", "pct")
        self.pct_base_capital = self.config.get("pct_base_capital", "initial")
        self.clip_max_abs_reward = self.config.get("clip_max_abs_reward", 1e5)
        self.reward_scaling_factor = self.config.get("reward_scaling_factor", 1.0)

        # Slippage & Sharpe
        self.slippage_pct = self.config.get("slippage_pct", 0.0)
        self.sharpe_reward_weight = self.config.get("sharpe_reward_weight", 0.0)

        # Misc gating
        self.max_drawdown_percent = self.config.get("max_drawdown_percent", 0.5)
        self.stop_on_end_of_data = self.config.get("stop_on_end_of_data", True)
        self.strict_capital_check = self.config.get("strict_capital_check", True)
        self.observation_window = self.config.get("observation_window", 30)
        self.use_risk_adjusted_logging = self.config.get("use_risk_adjusted_logging", True)

        # QUANTITY dict => { instrument_name: quantity }
        # Example: {"NIFTY": 75, "BANKNIFTY": 30}
        self.quantity_dict = self.config.get("quantity_dict", {})
        for inst in self.data_dict.keys():
            if inst not in self.quantity_dict:
                raise ValueError(f"quantity_dict missing entry for '{inst}'.")

        # ----------------
        # 2) INTERNAL TRACKERS
        # ----------------
        self.df = None
        self.max_steps = 0
        self.current_instrument = None
        self.current_timeframe = None

        self.capital = 0.0
        self.initial_capital = 0.0
        self.current_step = 0
        self.max_capital_seen = 0.0
        self.position = 0           # 0=flat, 1=long, -1=short
        self.entry_price = 0.0
        self.unrealized_pnl = 0.0
        self.wins = 0
        self.losses = 0
        self.hold_step_count = 0
        self.trade_durations = []

        # Logging
        self.capital_log = []
        self.win_percentage_log = []
        self.position_type_log = []
        self.step_returns = []
        self.max_drawdown_logged = []
        self.step_logs = []

        # Action space: 0=Hold, 1=Buy, 2=Sell
        self.action_space = gym.spaces.Discrete(3)
        # Observation space is defined at reset(), based on the actual DataFrame used

    # %%
    def reset(self):
        """
        Reset environment:
          1) Pick instrument/timeframe from config
          2) Load DataFrame
          3) Initialize capital, position, logs
          4) Return initial observation
        """
        self.current_instrument = self._select_instrument()
        self.current_timeframe = self._select_timeframe(self.current_instrument)

        # Access DataFrame
        self.df = self.data_dict[self.current_instrument][self.current_timeframe]
        self.max_steps = len(self.df)

        # Ensure "Target" & "StopLoss" columns exist
        if "Target" not in self.df.columns or "StopLoss" not in self.df.columns:
            raise ValueError(
                f"The DataFrame for {self.current_instrument}/{self.current_timeframe} "
                f"must contain 'Target' and 'StopLoss' columns."
            )

        # Initialize capital
        self.initial_capital = self._init_capital(self.df, self.initial_capital_multiplier)
        self.capital = self.initial_capital
        self.max_capital_seen = self.capital

        # Reset trackers
        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        self.unrealized_pnl = 0.0
        self.wins = 0
        self.losses = 0
        self.hold_step_count = 0
        self.trade_durations.clear()

        # Clear logs
        self.capital_log.clear()
        self.win_percentage_log.clear()
        self.position_type_log.clear()
        self.step_returns.clear()
        self.max_drawdown_logged.clear()
        self.step_logs.clear()

        # Brokerage cost
        self.brokerage_per_side = self.brokerage_dict[self.current_instrument]
        # Instrument quantity
        self.quantity = self.quantity_dict[self.current_instrument]

        # Define observation space shape after seeing chosen DF
        sample_obs = self._get_observation()
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=sample_obs.shape, dtype=np.float32
        )

        return sample_obs

    # %%
    def step(self, action):
        """
        For each step:
          1) Update unrealized PnL
          2) Take action with slippage
          3) Calculate realized PnL (if closing/flip)
          4) Incorporate "Target" or "StopLoss" logic if position is closed:
             - If realized >= 0 => realized = + (Target * quantity)
             - Else => realized = - (StopLoss * quantity)
          5) Compute step reward
          6) Check done conditions
          7) Log step info
          8) Return obs, reward, done, info
        """
        done = False
        step_reward = 0.0

        # If no more data, done
        if self.current_step >= self.max_steps:
            return self._get_observation(), 0.0, True, {}

        row_idx = min(self.current_step, self.max_steps - 1)
        current_price = self.df['close'].iloc[row_idx]

        # 1) Update unrealized PnL
        if self.position == 1:       # LONG
            # points captured so far in open position
            self.unrealized_pnl = (current_price - self.entry_price) * self.quantity
        elif self.position == -1:    # SHORT
            self.unrealized_pnl = (self.entry_price - current_price) * self.quantity
        else:
            self.unrealized_pnl = 0.0

        # Count how many steps in position
        if self.position != 0:
            self.hold_step_count += 1
        else:
            self.hold_step_count = 0

        # Track potential unrealized reward
        unreal_r = 0.0
        if self.track_unrealized_in_reward:
            unreal_r = self._transform_pnl_to_reward(self.unrealized_pnl)

        # 2) Action + slippage
        def buy_fill_price(ref_price):
            return ref_price * (1.0 + self.slippage_pct)

        def sell_fill_price(ref_price):
            return ref_price * (1.0 - self.slippage_pct)

        realized_pnl = 0.0
        basic_pnl_points = 0.0  # points without quantity

        # FLAT -> BUY or SELL
        if self.position == 0:
            if action == 1:  # BUY
                potential_fill = buy_fill_price(current_price)
                if self._can_open_position(potential_fill):
                    self.position = 1
                    self.entry_price = potential_fill
                    self.capital -= self.brokerage_per_side

            elif action == 2:  # SELL
                potential_fill = sell_fill_price(current_price)
                if self._can_open_position(potential_fill):
                    self.position = -1
                    self.entry_price = potential_fill
                    self.capital -= self.brokerage_per_side

        # LONG -> possibly SELL
        elif self.position == 1:
            if action == 2:
                exit_fill = sell_fill_price(current_price)
                basic_pnl_points = exit_fill - self.entry_price

                # Subtract brokerage for this exit
                self.capital -= self.brokerage_per_side

                # Check if basic PnL is profit or loss
                if basic_pnl_points >= 0:
                    # realized profit = Target * quantity
                    realized_pnl = self.df['Target'].iloc[row_idx] * self.quantity
                else:
                    # realized loss = -StopLoss * quantity
                    realized_pnl = -(self.df['StopLoss'].iloc[row_idx] * self.quantity)

                # Update capital
                self.capital += realized_pnl

                # Win/loss count
                if realized_pnl > 0:
                    self.wins += 1
                else:
                    self.losses += 1

                # Track hold time
                self.trade_durations.append(self.hold_step_count)

                if self.flip_position:
                    # Attempt to flip to short
                    flip_fill = sell_fill_price(current_price)
                    if self._can_open_position(flip_fill):
                        self.position = -1
                        self.entry_price = flip_fill
                        self.capital -= self.brokerage_per_side
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 1
                    else:
                        self.position = 0
                        self.entry_price = 0.0
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 0
                else:
                    # Close position
                    self.position = 0
                    self.entry_price = 0.0
                    self.unrealized_pnl = 0.0
                    self.hold_step_count = 0

        # SHORT -> possibly BUY
        elif self.position == -1:
            if action == 1:
                exit_fill = buy_fill_price(current_price)
                basic_pnl_points = self.entry_price - exit_fill

                # Subtract brokerage
                self.capital -= self.brokerage_per_side

                # Check if basic PnL is profit or loss
                if basic_pnl_points >= 0:
                    realized_pnl = self.df['Target'].iloc[row_idx] * self.quantity
                else:
                    realized_pnl = -(self.df['StopLoss'].iloc[row_idx] * self.quantity)

                # Update capital
                self.capital += realized_pnl

                # Win/loss count
                if realized_pnl > 0:
                    self.wins += 1
                else:
                    self.losses += 1

                # Track hold time
                self.trade_durations.append(self.hold_step_count)

                if self.flip_position:
                    # Attempt to flip to long
                    flip_fill = buy_fill_price(current_price)
                    if self._can_open_position(flip_fill):
                        self.position = 1
                        self.entry_price = flip_fill
                        self.capital -= self.brokerage_per_side
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 1
                    else:
                        self.position = 0
                        self.entry_price = 0.0
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 0
                else:
                    self.position = 0
                    self.entry_price = 0.0
                    self.unrealized_pnl = 0.0
                    self.hold_step_count = 0

        # 3) Transform realized PnL into reward
        realized_r = self._transform_pnl_to_reward(realized_pnl)

        # 4) Drawdown tracking
        self.max_capital_seen = max(self.max_capital_seen, self.capital)
        current_drawdown = (self.max_capital_seen - self.capital) / max(self.max_capital_seen, 1e-9)

        # 5) Sharpe approx
        base_step_reward = (realized_r + unreal_r * self.unrealized_reward_weight) * self.reward_scaling_factor

        # 6) Check terminal conditions
        if self.capital <= 0:
            done = True
        if self.max_drawdown_percent is not None and current_drawdown >= self.max_drawdown_percent:
            done = True
        if self.stop_on_end_of_data and (self.current_step + 1 >= self.max_steps):
            done = True

        # 7) Logging
        self.capital_log.append(self.capital)
        total_trades = self.wins + self.losses
        if total_trades > 0:
            win_percent = (self.wins / total_trades) * 100.0
        else:
            win_percent = 0.0
        self.win_percentage_log.append(win_percent)

        # Position string
        if self.position == 1:
            pos_str = f"Long-{self.current_instrument}-{self.current_timeframe}"
        elif self.position == -1:
            pos_str = f"Short-{self.current_instrument}-{self.current_timeframe}"
        else:
            pos_str = "Flat"

        self.position_type_log.append(pos_str)
        self.max_drawdown_logged.append(current_drawdown)

        # Step returns for approximate Sharpe
        if self.use_risk_adjusted_logging:
            if len(self.capital_log) > 1:
                prev_cap = self.capital_log[-2]
                step_ret = (self.capital - prev_cap) / max(abs(prev_cap), 1e-9)
            else:
                step_ret = 0.0
            self.step_returns.append(step_ret)

        sharpe_approx = 0.0
        if self.use_risk_adjusted_logging and len(self.step_returns) > 1:
            rets = np.array(self.step_returns)
            avg_r = rets.mean()
            std_r = rets.std() + 1e-9
            sharpe_approx = avg_r / std_r

        step_reward = base_step_reward + self.sharpe_reward_weight * sharpe_approx

        self.step_logs.append({
            "step": self.current_step,
            "capital": self.capital,
            "win_percent": win_percent,
            "position": pos_str,
            "unrealized_pnl": self.unrealized_pnl,
            "realized_pnl": realized_pnl,
            "points_captured_no_qty": basic_pnl_points,    # points (not multiplied)
            "drawdown_fraction": current_drawdown,
            "approx_sharpe": sharpe_approx,
            "reward": step_reward
        })

        # 8) Advance
        self.current_step += 1
        obs = self._get_observation()

        return obs, step_reward, done, {}

    # %%
    def render(self, mode='human'):
        """Optional debug info."""
        if len(self.capital_log) > 0:
            idx = len(self.capital_log) - 1
            cap_str = f"{self.capital_log[idx]:.2f}"
            winp_str = f"{self.win_percentage_log[idx]:.2f}"
            pos_str = self.position_type_log[idx]
            dd_str = f"{self.max_drawdown_logged[idx]*100:.2f}%"
            print(
                f"Step: {self.current_step}, "
                f"Cap: {cap_str}, "
                f"Win%: {winp_str}, "
                f"Pos: {pos_str}, "
                f"DD: {dd_str}"
            )
        else:
            print(f"Step: {self.current_step}, Cap: {self.capital:.2f}, Pos: [No log yet]")

    # %%
    def close(self):
        """Placeholder for cleanup, if needed."""
        pass

    # %%
    # ----------------------------------------------------------------
    # EVALUATION / LOGGING
    # ----------------------------------------------------------------
    def get_evaluation_report(self):
        """Return a dictionary of final performance metrics and logs."""
        if self.capital_log:
            final_capital = self.capital_log[-1]
        else:
            final_capital = self.capital

        net_profit = final_capital - self.initial_capital
        if self.win_percentage_log:
            final_winrate = self.win_percentage_log[-1]
        else:
            final_winrate = 0.0

        if self.max_drawdown_logged:
            max_dd = max(self.max_drawdown_logged)
        else:
            max_dd = 0.0

        report = {
            "instrument": self.current_instrument,
            "timeframe": self.current_timeframe,
            "initial_capital": self.initial_capital,
            "final_capital": final_capital,
            "net_profit": net_profit,
            "final_winrate_percent": final_winrate,
            "capital_log": self.capital_log.copy(),
            "winrate_each_step": self.win_percentage_log.copy(),
            "position_each_step": self.position_type_log.copy(),
            "drawdown_each_step": self.max_drawdown_logged.copy(),
            "max_drawdown_fraction": max_dd,
            "step_logs": self.step_logs.copy()
        }

        # Approximate Sharpe
        if self.use_risk_adjusted_logging and len(self.step_returns) > 1:
            rets = np.array(self.step_returns)
            avg_r = rets.mean()
            std_r = rets.std() + 1e-9
            sharpe_approx = avg_r / std_r
            report["approx_sharpe"] = sharpe_approx
        else:
            report["approx_sharpe"] = 0.0

        return report

    # %%
    def get_step_logs_df(self):
        """Return a DataFrame of step logs for analysis."""
        return pd.DataFrame(self.step_logs)

    # %%
    # ----------------------------------------------------------------
    # INTERNAL HELPERS
    # ----------------------------------------------------------------
    def _select_instrument(self):
        """Pick instrument from the top-level keys of data_dict."""
        instruments = list(self.data_dict.keys())
        mode = self.instrument_selection

        if mode == "random":
            return random.choice(instruments)
        elif mode == "sequential":
            if not hasattr(self, '_seq_inst_idx'):
                self._seq_inst_idx = 0
            chosen = instruments[self._seq_inst_idx % len(instruments)]
            self._seq_inst_idx += 1
            return chosen
        else:
            # must be a string that matches a known instrument
            if mode not in instruments:
                raise ValueError(f"instrument_selection='{mode}' not found in data_dict.")
            return mode

    # %%
    def _select_timeframe(self, instrument):
        """Pick timeframe from the sub-dict data_dict[instrument]."""
        timeframes = list(self.data_dict[instrument].keys())
        mode = self.timeframe_selection

        if mode == "random":
            return random.choice(timeframes)
        elif mode == "sequential":
            if not hasattr(self, '_seq_tf_idx'):
                self._seq_tf_idx = 0
            chosen = timeframes[self._seq_tf_idx % len(timeframes)]
            self._seq_tf_idx += 1
            return chosen
        else:
            if mode not in timeframes:
                raise ValueError(f"timeframe_selection='{mode}' not found in data_dict[{instrument}].")
            return mode

    # %%
    def _init_capital(self, df, multiplier):
        """Initialize capital as multiplier * max(close)."""
        max_close = df['close'].max()
        return multiplier * max_close

    # %%
    def _can_open_position(self, fill_price):
        """
        Check if we have enough capital to open a position + pay brokerage.
        If strict_capital_check=False, skip this check.
        """
        if not self.strict_capital_check:
            return True
        # Here, consider only the fill_price for 1 unit + brokerage,
        # because the actual PnL will multiply by quantity once closed.
        cost_to_open = fill_price + self.brokerage_per_side
        if self.capital - cost_to_open <= 0:
            return False
        return True

    # %%
    def _transform_pnl_to_reward(self, pnl):
        """
        Convert a PnL value (realized or unrealized) to a reward according to `reward_mode`:
         - "raw": returns pnl as-is
         - "pct": returns (pnl / chosen base capital)
         - "log": sign(pnl) * log(1 + abs(pnl))
         - "clip": clip(pnl) to +/- clip_max_abs_reward
        """
        if self.reward_mode == "raw":
            r = pnl
        elif self.reward_mode == "pct":
            if self.pct_base_capital == "current":
                base_cap = max(self.capital, 1e-9)
            elif self.pct_base_capital == "initial":
                base_cap = max(self.initial_capital, 1e-9)
            elif self.pct_base_capital == "max":
                base_cap = max(self.max_capital_seen, 1e-9)
            else:
                base_cap = max(self.initial_capital, 1e-9)
            r = pnl / base_cap
        elif self.reward_mode == "log":
            sgn = np.sign(pnl)
            mag = abs(pnl)
            r = sgn * np.log(1.0 + mag)
        elif self.reward_mode == "clip":
            r = np.clip(pnl, -self.clip_max_abs_reward, self.clip_max_abs_reward)
        else:
            r = pnl
        return r

    # %%
    def _get_observation(self):
        """
        Return the entire row of the DataFrame (all columns) at the current step,
        cast to float32. This includes any custom features in your DataFrame.

        IMPORTANT: If your DataFrame has any non-numeric columns,
        you'll need to remove or transform them before converting to float32.
        """
        row_idx = min(self.current_step, self.max_steps - 1)
        row_values = self.df.iloc[row_idx].values  # all columns
        obs = row_values.astype(np.float32)        # ensure float32
        return obs

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [60]:
config = {
    "data_dict": {
        "Nifty": {
            "2m": df_nifty_2m,
            "5m": df_nifty_5m,
            "15m": df_nifty_15m
        },
        "BankNifty": {
            "2m": df_bnf_2m,
            "5m": df_bnf_5m,
            "15m": df_bnf_15m
        }
    }, # { "instrument_name": pd.DataFrame, ... } - user sets externally
    "brokerage_dict": {
        "Nifty": 20.0,
        "BankNifty": 20.0},               # Add brokerage specific for each instrument
    "quantity_dict": {
        "Nifty": 75,
        "BankNifty": 30
    },
    "instrument_selection": "sequential",     # "random", "sequential", or a specific instrument name
    "timeframe_selection": "sequential",
    "initial_capital_multiplier": 3.0,    # Env capital = multiplier * max(close)
    "flip_position": True,                # if True, Sell on Long will close & flip to Short in one step
    "track_unrealized_in_reward": True,   # add incremental reward for open positions
    "unrealized_reward_weight": 0.3,      # weight for unrealized PnL in each step's reward

    # Reward mode: "raw", "pct", "log", or "clip"
    "reward_mode": "pct",
    "pct_base_capital": "initial",        # "current" or "initial" for 'pct'/'log' modes
    "clip_max_abs_reward": 1e5,           # used if reward_mode="clip"
    "reward_scaling_factor": 1.0,         # scale the realized + unrealized portion of reward

    # Slippage & Sharpe config
    "slippage_pct": 0.0002,               # e.g. 0.02% slippage
    "sharpe_reward_weight": 0.1,          # how strongly Sharpe ratio is factored into step reward

    # We keep max_drawdown_percent only to end episodes, if desired.
    # (No longer used for reward penalty, since we replaced drawdown with Sharpe.)
    "max_drawdown_percent": 0.5,          # end episode if drawdown >= 50% (None to disable)
    "stop_on_end_of_data": True,          # end episode when we pass last row of the chosen instrument
    "strict_capital_check": True,         # disallow new positions if capital < (price + brokerage)
    "observation_window": 30,             # how many past bars to include in each observation

    "use_risk_adjusted_logging": True     # track step returns so we can compute approximate Sharpe
}

In [61]:
env = SingleInstrumentTradingEnv(config)
obs = env.reset()

done = False
while not done:
    action = env.action_space.sample()  # or from your RL policy
    obs, reward, done, info = env.step(action)
    env.render()

Step: 1, Cap: 78820.35, Win%: 0.00, Pos: Flat, DD: 0.00%
Step: 2, Cap: 78800.35, Win%: 0.00, Pos: Long-Nifty-2m, DD: 0.03%
Step: 3, Cap: 81275.10, Win%: 100.00, Pos: Short-Nifty-2m, DD: 0.00%
Step: 4, Cap: 81275.10, Win%: 100.00, Pos: Short-Nifty-2m, DD: 0.00%
Step: 5, Cap: 79592.60, Win%: 50.00, Pos: Long-Nifty-2m, DD: 2.07%
Step: 6, Cap: 77923.60, Win%: 33.33, Pos: Short-Nifty-2m, DD: 4.12%
Step: 7, Cap: 77923.60, Win%: 33.33, Pos: Short-Nifty-2m, DD: 4.12%
Step: 8, Cap: 77923.60, Win%: 33.33, Pos: Short-Nifty-2m, DD: 4.12%
Step: 9, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD: 6.04%
Step: 10, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD: 6.04%
Step: 11, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD: 6.04%
Step: 12, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD: 6.04%
Step: 13, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD: 6.04%
Step: 14, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD: 6.04%
Step: 15, Cap: 76366.35, Win%: 25.00, Pos: Long-Nifty-2m, DD:

In [62]:
# After the episode, you can compute stats like approximate Sharpe ratio
if config["use_risk_adjusted_logging"]:
    rets = np.array(env.step_returns)
    avg_r = rets.mean()
    std_r = rets.std()
    sharpe_approx = avg_r / std_r if std_r > 1e-9 else 0
    print("Approx Sharpe:", sharpe_approx)

Approx Sharpe: -0.023798812980318683


In [63]:
report = env.get_evaluation_report()

print("Instrument:", report["instrument"])
print("Initial Capital:", report["initial_capital"])
print("Final Capital:", report["final_capital"])
print("Net Profit:", report["net_profit"])
print("Final Win Rate (%):", report["final_winrate_percent"])
print("Max Drawdown (%):", report["max_drawdown_fraction"] * 100.0)

Instrument: Nifty
Initial Capital: 78820.35
Final Capital: 53263.850000000006
Net Profit: -25556.5
Final Win Rate (%): 29.767441860465116
Max Drawdown (%): 50.14958255866898


In [64]:
step_df = env.get_step_logs_df()
step_df

,step,capital,win_percent,position,unrealized_pnl,realized_pnl,points_captured_no_qty,drawdown_fraction,approx_sharpe,reward
0,0,78820.35,0.000000,Flat,0.0000,0.00,0.00000,0.000000,0.000000,0.000000
1,1,78800.35,0.000000,Long-Nifty-2m,0.0000,0.00,0.00000,0.000254,-0.999992,-0.099999
2,2,81275.10,100.000000,Short-Nifty-2m,0.0000,2514.75,11.41028,0.000000,0.698555,0.105987
3,3,81275.10,100.000000,Short-Nifty-2m,810.2490,0.00,0.00000,0.000000,0.571131,0.060197
4,4,79592.60,50.000000,Long-Nifty-2m,0.0000,-1642.50,-7.39348,0.020701,0.125215,-0.009458
...,...,...,...,...,...,...,...,...,...,...
622,622,54752.10,29.906542,Long-Nifty-2m,-2461.6395,0.00,0.00000,0.487567,-0.021610,-0.011530
623,623,54752.10,29.906542,Long-Nifty-2m,-3436.6395,0.00,0.00000,0.487567,-0.021592,-0.015240
624,624,54752.10,29.906542,Long-Nifty-2m,-2769.1395,0.00,0.00000,0.487567,-0.021575,-0.012697
625,625,54752.10,29.906542,Long-Nifty-2m,-3076.6395,0.00,0.00000,0.487567,-0.021558,-0.013866


In [65]:
# %%
# =============================================
# 1) SETUP AND CONFIG
#    (Assuming you have your SingleInstrumentTradingEnv already defined)
# =============================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

# For logging / DataFrame
import pandas as pd
from collections import deque

# If you have CUDA available, we use it. Otherwise, CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -------------------------------
# 1b) Model Configuration
# -------------------------------
model_config = {
    # Sequence handling
    "seq_length": 16,          # We'll buffer the last 16 observations
    # TCN part
    "tcn_channels": [64, 64],  # Example: two Conv1d layers with 64 filters each
    "kernel_size": 3,
    "dropout": 0.1,
    # Transformer part
    "d_model": 64,             # dimension in the Transformer
    "nhead": 4,                # number of attention heads
    "num_layers": 2,           # number of TransformerEncoder layers
    "transformer_dropout": 0.1,
    # Output dimension (for discrete actions: 3)
    "num_actions": 3,
    # Learning rate, etc.
    "lr": 1e-4,
    "gamma": 0.99,             # discount factor
    "batch_size": 32,
    "update_steps": 512,       # how many steps per update
    "epochs": 10,              # how many times to go over the collected batch
}

# (Below we'll define a custom policy network that uses a TCN + Transformer.)


# %%
# =======================================================
# 2) MODEL ARCHITECTURE (TRANSFORMER + TCN) FOR RL POLICY
# =======================================================
class TCNBlock(nn.Module):
    """
    A simple TCN block that applies causal convolutions.
    We'll assume the input shape is (N, C, L) => (batch, channels, length).
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, dropout=0.1):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size, padding=(kernel_size - 1),
            bias=True, padding_mode='zeros'
        )
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None

    def forward(self, x):
        # x: (batch, channels, length)
        out = self.conv(x)[:, :, :- (self.conv.kernel_size[0] - 1)]  # causal shift
        out = self.relu(out)
        out = self.dropout(out)
        if self.residual is not None:
            # match shapes for residual
            res = self.residual(x)
            res = res[:, :, :out.shape[2]]  # match length
            out = out + res
        else:
            out = out + x[:, :, :out.shape[2]]
        return out


class TransformerTCNNet(nn.Module):
    """
    Combines a TCN to extract local temporal patterns,
    then a TransformerEncoder to capture longer-range dependencies.
    Finally outputs policy logits and value for discrete action RL.
    """
    def __init__(self, obs_dim, model_cfg):
        super().__init__()
        self.seq_length = model_cfg["seq_length"]
        tcn_channels = model_cfg["tcn_channels"]
        kernel_size = model_cfg["kernel_size"]
        dropout = model_cfg["dropout"]

        d_model = model_cfg["d_model"]
        nhead = model_cfg["nhead"]
        num_layers = model_cfg["num_layers"]
        transformer_dropout = model_cfg["transformer_dropout"]

        num_actions = model_cfg["num_actions"]

        # We treat obs_dim as "channels" for TCN input,
        # but typically TCN expects (N, C, L). So we map obs => (N, obs_dim, seq_len).
        # We'll build TCN with a stack of TCNBlock.

        layers = []
        in_channels = obs_dim
        for out_channels in tcn_channels:
            layers.append(TCNBlock(in_channels, out_channels, kernel_size, dropout))
            in_channels = out_channels
        self.tcn = nn.Sequential(*layers)

        # For the Transformer, we will feed (seq_len, batch, d_model).
        # So we map TCN output => shape (batch, seq_len, channels).
        # Then we project channels => d_model.
        self.proj_to_dmodel = nn.Linear(tcn_channels[-1], d_model)

        # Create a TransformerEncoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=transformer_dropout,
            batch_first=False  # we'll rearrange dims to (seq, batch, d_model)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Policy & Value heads
        self.policy_head = nn.Linear(d_model, num_actions)
        self.value_head = nn.Linear(d_model, 1)

    def forward(self, x):
        """
        x shape: (batch, seq_length, obs_dim)
        1) swap to (batch, obs_dim, seq_length) for TCN
        2) pass TCN => (batch, out_channels, new_length)
        3) swap to (batch, new_length, out_channels)
        4) project => (batch, new_length, d_model)
        5) transform dims => (new_length, batch, d_model) for Transformer
        6) pass through Transformer => same shape
        7) take last time-step => (batch, d_model)
        8) policy_head => (batch, num_actions), value_head => (batch, 1)
        """
        bsz = x.shape[0]
        # 1) For TCN, we interpret obs_dim as channels:
        x_tcn = x.permute(0, 2, 1)  # now shape (batch, obs_dim, seq_length)

        # 2) Pass through TCN
        tcn_out = self.tcn(x_tcn)  # shape (batch, out_channels, seq_length_after)

        # (Because of the causal shift, seq_length_after might be smaller, typically = seq_length)
        seq_len_out = tcn_out.shape[2]

        # 3) Move channels to last dimension
        tcn_out = tcn_out.permute(0, 2, 1)  # (batch, seq_len_out, out_channels)

        # 4) Project to d_model
        d_model_out = self.proj_to_dmodel(tcn_out)  # (batch, seq_len_out, d_model)

        # 5) Rearrange for Transformer => (seq_len_out, batch, d_model)
        d_model_out = d_model_out.permute(1, 0, 2)

        # 6) Pass through Transformer
        trans_out = self.transformer_encoder(d_model_out)  # shape (seq_len_out, batch, d_model)

        # 7) Take last time-step
        # We'll assume the last time-step is trans_out[-1], shape (batch, d_model)
        last_step = trans_out[-1, :, :]  # (batch, d_model)

        # 8) Policy & Value
        policy_logits = self.policy_head(last_step)  # (batch, num_actions)
        value = self.value_head(last_step)           # (batch, 1)

        return policy_logits, value.squeeze(-1)


# %%
# =====================================================
# 3) A SIMPLE PPO-LIKE AGENT THAT USES THE ABOVE MODEL
# =====================================================
class SimplePPOAgent:
    """
    A simplified PPO-like agent that:
      - Collects rollouts of size model_cfg["update_steps"]
      - Uses GAE or simple advantage estimate
      - Optimizes for a few epochs
      - Then repeats
    """
    def __init__(self, env, model_cfg):
        self.env = env
        self.model_cfg = model_cfg

        # We find obs_dim from a sample. If your env returns entire row, shape = (num_features,).
        sample_obs = env.reset()
        env.reset()  # just to be safe, discard
        self.obs_dim = sample_obs.shape[0]
        self.num_actions = model_cfg["num_actions"]

        self.seq_length = model_cfg["seq_length"]
        self.gamma = model_cfg["gamma"]
        self.lr = model_cfg["lr"]
        self.batch_size = model_cfg["batch_size"]
        self.update_steps = model_cfg["update_steps"]
        self.epochs = model_cfg["epochs"]

        # Buffer for the last N observations so we can feed the TCN + Transformer
        self.obs_buffer = deque(maxlen=self.seq_length)

        # Build policy network
        self.network = TransformerTCNNet(self.obs_dim, model_cfg).to(device)
        # Create an optimizer
        self.optimizer = optim.Adam(self.network.parameters(), lr=self.lr)

        # PPO clipping param, etc.
        self.clip_epsilon = 0.2
        self.value_loss_coef = 0.5
        self.entropy_coef = 0.01

    def _reset_obs_buffer(self, obs):
        """Fill the buffer with the initial obs repeated, so sequence is not random at first."""
        self.obs_buffer.clear()
        for _ in range(self.seq_length):
            self.obs_buffer.append(obs)

    def get_policy_and_value(self, obs_seq):
        """
        obs_seq shape => (seq_length, obs_dim).
        We'll reshape to (1, seq_length, obs_dim) for the network.
        """
        x_t = torch.FloatTensor(obs_seq).unsqueeze(0).to(device)
        logits, v = self.network(x_t)  # (1, num_actions), (1)
        return logits[0], v[0]

    def act(self, obs_seq):
        """
        Return an action by sampling from the policy distribution.
        """
        with torch.no_grad():
            logits, value = self.get_policy_and_value(obs_seq)
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
        return action.item(), dist.log_prob(action), dist.entropy().item(), value.item()

    def compute_returns_and_advantage(self, rewards, values, dones, last_value):
        """
        GAE-lambda or simple advantage. We'll do a simple advantage for clarity:
            V-target (bootstrap) = r_t + gamma * V_{t+1} (if not done)
        Then advantage = V-target - V_{t}
        """
        returns = []
        advs = []
        gae = 0.0

        # We have T steps, plus 1 last_value
        for step in reversed(range(len(rewards))):
            if dones[step]:
                delta = rewards[step] - values[step]
                gae = delta
            else:
                delta = rewards[step] + self.gamma * last_value - values[step]
                gae = delta
            returns.append(values[step] + gae)
            advs.append(gae)
            last_value = values[step]
        returns.reverse()
        advs.reverse()
        return returns, advs

    def update(self, rollout):
        """
        rollout is a list of dictionaries with:
          {
            'obs_seq': np.array of shape (seq_length, obs_dim),
            'action': int,
            'log_prob': float,
            'value': float,
            'reward': float,
            'done': bool
          }
        plus the last_value from the final state
        """
        # 1) separate arrays
        obs_seqs = []
        actions = []
        old_log_probs = []
        values = []
        rewards = []
        dones = []
        for r in rollout:
            obs_seqs.append(r['obs_seq'])
            actions.append(r['action'])
            old_log_probs.append(r['log_prob'])
            values.append(r['value'])
            rewards.append(r['reward'])
            dones.append(r['done'])

        # last value
        last_value = rollout[-1]["last_value"]

        # 2) compute returns and advantage
        returns, advantages = self.compute_returns_and_advantage(
            rewards, values, dones, last_value
        )
        advantages = np.array(advantages, dtype=np.float32)
        returns = np.array(returns, dtype=np.float32)

        # 3) training in mini-batches
        obs_seqs = np.array(obs_seqs, dtype=np.float32)   # shape (T, seq_length, obs_dim)
        actions = np.array(actions, dtype=np.int64)       # shape (T,)
        old_log_probs = np.array(old_log_probs, dtype=np.float32)
        values = np.array(values, dtype=np.float32)

        dataset_size = len(rollout)
        indices = np.arange(dataset_size)

        for epoch in range(self.epochs):
            np.random.shuffle(indices)
            for start_idx in range(0, dataset_size, self.batch_size):
                end_idx = start_idx + self.batch_size
                batch_idx = indices[start_idx:end_idx]

                batch_obs_seq = torch.FloatTensor(obs_seqs[batch_idx]).to(device)
                batch_actions = torch.LongTensor(actions[batch_idx]).to(device)
                batch_old_log_probs = torch.FloatTensor(old_log_probs[batch_idx]).to(device)
                batch_adv = torch.FloatTensor(advantages[batch_idx]).to(device)
                batch_returns = torch.FloatTensor(returns[batch_idx]).to(device)

                # Forward pass
                logits, v_preds = self.network(batch_obs_seq)  # (B, num_actions), (B)
                dist = torch.distributions.Categorical(logits=logits)
                new_log_probs = dist.log_prob(batch_actions)
                entropy = dist.entropy().mean()

                # Ratio for PPO
                ratio = torch.exp(new_log_probs - batch_old_log_probs)
                surr1 = ratio * batch_adv
                surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * batch_adv
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss
                value_loss = 0.5 * (batch_returns - v_preds).pow(2).mean()

                # Combine
                loss = policy_loss + self.value_loss_coef * value_loss - self.entropy_coef * entropy

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

    def train_one_iteration(self):
        """
        Collect self.update_steps transitions, then do PPO update.
        Return logs for analysis.
        """
        rollout = []
        obs = self.env.reset()
        self._reset_obs_buffer(obs)
        done = False

        step_count = 0
        # We'll track a single rollout of length update_steps
        while step_count < self.update_steps:
            obs_seq = np.array(self.obs_buffer, dtype=np.float32)  # shape (seq_length, obs_dim)
            action, log_prob, ent, value = self.act(obs_seq)

            next_obs, reward, done, info = self.env.step(action)
            # We store the transition
            transition = {
                "obs_seq": obs_seq.copy(),
                "action": action,
                "log_prob": log_prob,
                "value": value,
                "reward": reward,
                "done": done,
            }

            # We'll get next state's value for advantage calc
            if done:
                # If done, the "last_value" is 0
                # (or you can do something more advanced if it's a forced done)
                transition["last_value"] = 0.0
                rollout.append(transition)
                # We break the episode here, but for PPO, we might continue collecting from a new episode
                # if step_count < update_steps.
                obs = self.env.reset()
                self._reset_obs_buffer(obs)
                done = False
            else:
                # not done
                # We do a quick forward pass to get the next state value, but won't store that as we want
                # to keep consistent. We'll just store it upon the NEXT transition iteration or after loop.
                # We'll fill it after the loop.
                transition["last_value"] = None
                rollout.append(transition)
                obs = next_obs
                self.obs_buffer.append(next_obs)

            step_count += 1

        # Now fill the 'last_value' for any transition that had None
        if not done:
            # if not done at the end, we do a forward pass on the final state
            obs_seq = np.array(self.obs_buffer, dtype=np.float32)
            with torch.no_grad():
                _, final_value = self.get_policy_and_value(obs_seq)
            last_val = final_value.item()
        else:
            last_val = 0.0

        for r in reversed(rollout):
            if r["last_value"] is None:
                r["last_value"] = last_val
            else:
                break

        # Perform the PPO update
        self.update(rollout)

        # We can get the environment step logs and final logs if desired
        step_logs_df = self.env.get_step_logs_df()
        # We'll also extract the final environment stats if the env provides them
        # (But note: if we keep resetting in the middle of the rollout, the logs represent partial episodes.)
        # For a stable measure, you might want to only measure logs after a full episode is done.

        # Return any logs for external usage
        return {
            "step_logs_df": step_logs_df
        }

# %%
# =====================================================
# 4) TRAINING LOOP & LOGGING
# =====================================================
def train_agent(env, agent, total_iterations=10):
    """
    Run multiple iterations, each collecting 'update_steps' transitions,
    then performing PPO update. Log environment info each iteration.
    """
    all_logs = []
    for iteration in range(total_iterations):
        result = agent.train_one_iteration()

        # Grab environment logs each iteration
        step_logs_df = result["step_logs_df"]

        # End-of-iteration stats (like final capital, final win rate, etc.)
        # The environment might have multiple resets in one iteration (if done is True early).
        # So let's also fetch the environment's "get_evaluation_report()" if that suits you.
        final_report = env.get_evaluation_report()

        iteration_log = {
            "iteration": iteration,
            "final_capital": final_report["final_capital"],
            "final_winrate": final_report["final_winrate_percent"],
            "net_profit": final_report["net_profit"],
            "approx_sharpe": final_report["approx_sharpe"],
            "step_logs_df_shape": step_logs_df.shape,
        }
        all_logs.append(iteration_log)

        print(f"Iteration {iteration} | "
              f"FinalCap: {iteration_log['final_capital']:.2f}, "
              f"WinRate: {iteration_log['final_winrate']:.2f}%, "
              f"NetProfit: {iteration_log['net_profit']:.2f}, "
              f"SharpeApprox: {iteration_log['approx_sharpe']:.2f}")

    return pd.DataFrame(all_logs)

# %%
# ===========================================
# 5) EXAMPLE USAGE AND LOGGING FINAL RESULTS
# ===========================================
# Uncomment below to run everything end-to-end:

# 1) Create environment
env = SingleInstrumentTradingEnv(env_config)

# 2) Create agent
agent = SimplePPOAgent(env, model_config)

# 3) Train
training_logs_df = train_agent(env, agent, total_iterations=20)

# 4) Check final logs
print("\nTraining Summary:")

NameError: name 'env_config' is not defined

In [ ]:
training_logs_df

Version 2

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": nf_index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [ ]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)